# 🐍 Snake Game com IA treinada em PyTorch

Este notebook implementa:
1. **Jogo da Cobrinha** — ambiente completo com renderização visual
2. **Rede Neural em PyTorch** — agente DQN (Deep Q-Network) que aprende a jogar
3. **Treinamento com feedback visual** — gráficos de reward, epsilon, gameplay
4. **Documentação completa** — cada seção explicada para quem nunca viu IA

---

## 🧠 Conceitos Fundamentais de IA (para quem é de Web Dev)

### O que é Machine Learning?

Pense em ML como a diferença entre **programar regras** vs **ensinar por exemplo**.

- **Programação tradicional**: você escreve cada regra manualmente.
  ```javascript
  // Exemplo: validar email com regras explícitas
  if (!email.includes('@')) return 'Email inválido';
  ```
- **Machine Learning**: você dá exemplos e o computador **descobre as regras sozinho**.
  ```
  // Você dá 1000 emails válidos e 1000 inválidos.
  // A IA descobre sozinha a 'regra' de o que é um email válido.
  ```

### O que é uma Rede Neural?

Pense numa rede neural como uma **função matemática sofisticada**.

- No backend, você faz: `result = fn(input)` — a lógica dentro de `fn` é fixa.
- Numa rede neural: `result = rede(input)` — a lógica dentro de `rede` é **aprendida** através de exemplos.

**Analogia com JavaScript:**
```javascript
// Programação normal: você define a lógica
function preverNota(credito, historico) {
    return (credito * 0.7) + (historico * 0.3);  // pesos fixos: 0.7 e 0.3
}

// Rede neural: a IA DESCUBRE os pesos (0.7, 0.3) sozinha
// Os pesos começam aleatórios e são ajustados com o treino
```

### O que é Deep Learning?

"Deep" (profundo) significa que a rede neural tem **múltiplas camadas** de processamento. É como ter vários `pipe()` no Express — cada camada transforma os dados.

```
Input → [Camada 1] → [Camada 2] → [Camada 3] → Output
         (feature      (combina      (refina       (resultado
          extraction)   features)     padrões)     final)
```

### O que é Reinforcement Learning (RL)?

**Analogia: treinar um cachorro.**
- Você NÃO diz ao cachorro exatamente o que fazer.
- Quando ele faz algo bom → **recompensa** (petisco 🦴)
- Quando faz algo ruim → **punição** (bronca 😠)
- Com o tempo, o cachorro aprende o que fazer através do **trial and error**.

No nosso caso:
- O **cachorro** = a IA
- O **truque** = jogar Snake
- Os **petiscos** = recompensas (+10 pontos, +0.1 por sobreviver)
- As **broncas** = punições (-10 por morrer)

### Glossário Rápido

| Termo | Significado | Analogia Web Dev |
|---|---|---|
| **Agente** | A IA que toma decisões | O "código" que decide para onde a cobra vai |
| **Ambiente** | O mundo onde o agente age | O servidor/game engine |
| **Estado (State)** | A "foto" da situação atual | O `req.body` — dados de entrada |
| **Ação (Action)** | O que o agente decide fazer | O handler do `POST /move` |
| **Reward** | Feedback (bom/ruim) da ação | O `res.status()` — 200 (bom) ou 500 (ruim) |
| **Episódio** | Uma partida completa (do start ao game over) | Uma request completa do início ao fim |
| **Epoch/Treino** | Muitos episódios rodados | Rodar testes múltiplas vezes |
| **Loss** | Quão errado o modelo está | Métrica de erro que queremos minimizar |
| **Backpropagation** | Como a rede "aprende" (ajusta pesos) | Como o ajuste de pesos é propagado para trás |
| **Pesos (Weights)** | Números internos que definem o comportamento | O `config.js` que determina o comportamento do app |

In [ ]:
# ============================================================
# IMPORTS — Bibliotecas que vamos usar
# ============================================================
# Imports nativos do Python
import random            # escolhas aleatórias (como Math.random())
import time              # medição de tempo
from collections import deque  # fila com tamanho máximo (buffer circular)

# Computação numérica — arrays eficientes (como TypedArrays em JS)
import numpy as np

# Visualização — gráficos
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# PyTorch — framework de Deep Learning ("React" do mundo da IA)
import torch
import torch.nn as nn            # módulos de rede neural
import torch.optim as optim      # otimizadores (ajustam os pesos)
import torch.nn.functional as F  # funções de ativação

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

# Parte 1: Ambiente do Jogo Snake

## O Conceito: Ambiente = O "Servidor" do Jogo

Em Reinforcement Learning, o **ambiente** é tudo que está ao redor da IA:

```
┌──────────────────────────────────────────┐
│                  LOOP DO JOGO             │
│                                           │
│   ┌─────┐  ação (0,1,2,3)   ┌──────────┐ │
│   │  IA ├──────────────────►│ Ambiente │ │
│   └─────┘                   │  (jogo)  │ │
│      ▲                      └──────────┘ │
│      │                         │          │
│      │  (novo estado, reward,  │          │
│      │   game over?)           │          │
│      └─────────────────────────┘          │
└──────────────────────────────────────────┘
```

A IA **não vê o tabuleiro inteiro**. Ela só recebe um vetor de números (o **estado**) que descreve a situação. É como um **endpoint de API** — você recebe um JSON com dados, não a imagem da tela.

## O Estado (State) — 10 Números que Descrevem Tudo

Em vez de dar à IA uma imagem do tabuleiro (que seriam 100 pixels = 100 inputs), damos 10 números estratégicos:

| Índice | O que representa | Exemplo valor | Explicação |
|---|---|---|---|
| 0 | Posição X da cabeça | 0.5 | Normalizado (0 = borda esquerda, 1 = borda direita) |
| 1 | Posição Y da cabeça | 0.3 | Normalizado (0 = topo, 1 = fundo) |
| 2 | Direção X | 1.0 | Indo para direita = 1, esquerda = -1 |
| 3 | Direção Y | 0.0 | Não subindo/descendo = 0 |
| 4 | Comida relativa X | 0.2 | Comida está 0.2 (20% do tabuleiro) à direita |
| 5 | Comida relativa Y | -0.1 | Comida está 0.1 acima |
| 6 | Perigo em frente? | 1.0 | Tem parede ou corpo na frente? (sim=1, não=0) |
| 7 | Perigo à direita? | 0.0 | Tem parede ou corpo à direita? |
| 8 | Perigo à esquerda? | 0.0 | Tem parede ou corpo à esquerda? |
| 9 | Comprimento da cobra | 0.03 | 3 segmentos de 100 células totais = 3% |

## O Sistema de Recompensas (Rewards)

É assim que "falamos" com a IA. Ela não sabe as regras do Snake — só sabe o que gera pontos:

| Evento | Reward | Por quê? |
|---|---|---|
| Comeu comida | **+10** | Objetivo principal! |
| Morreu | **-10** | Punição por bater |
| Passos sem morrer | **+0.1** | Incentiva sobreviver (mas menos que comer) |
| Ficou muito tempo sem comer | **-5** | Penaliza ficar girando em loop |

**Analogia:** É como dar XP num RPG. O jogador (IA) não sabe o que fazer no início, mas vai percebendo que certas ações dão mais XP que outras.

In [ ]:
# ============================================================
# SNAKE GAME — Ambiente
# ============================================================
class SnakeGame:
    """
    Ambiente do jogo Snake.
    
    Pense nisso como o 'game engine' ou o 'servidor' do jogo.
    A IA interage com ele através do método .step(action).
    
    Ações (como endpoints de uma API):
        0 — mover para cima
        1 — mover para baixo
        2 — mover para esquerda
        3 — mover para direita
    
    Resposta (como um JSON de resposta):
        state  — vetor com 10 números descrevendo a situação
        reward — número dado como feedback (XP)
        done   — boolean, se o jogo acabou
    """

    def __init__(self, grid_size=10):
        self.GRID_SIZE = grid_size
        self.reset()

    def reset(self):
        """
        Reinicia o jogo — equivale a um 'novo jogo / restart'.
        Cobra começa no centro, tamanho 3, indo para a direita.
        Retorna o estado inicial para a IA decidir a primeira ação.
        """
        cx, cy = self.GRID_SIZE // 2, self.GRID_SIZE // 2
        self.snake = [(cx, cy), (cx - 1, cy), (cx - 2, cy)]
        self.direction = (1, 0)  # indo para direita
        self.score = 0
        self.food = self._place_food()
        self.steps = 0
        self.steps_without_food = 0
        self.game_over = False
        return self._get_state()

    def _place_food(self):
        """Coloca comida em posição aleatória livre na grid."""
        while True:
            pos = (random.randint(0, self.GRID_SIZE - 1),
                   random.randint(0, self.GRID_SIZE - 1))
            if pos not in self.snake:
                return pos

    def step(self, action):
        """
        Executa UMA ação no jogo (um 'frame' do jogo).
        
        Parâmetro:
            action: int de 0-3 (cima, baixo, esquerda, direita)
        
        Retorna (estado, reward, done):
            estado  — como está o jogo APÓS a ação
            reward  — feedback numérico para a IA
            done    — se o jogo acabou (game over)
        """
        # Mapeia número da ação para direção (x, y)
        action_map = {
            0: (0, -1),   # cima    (y diminui no topo)
            1: (0, 1),    # baixo   (y aumenta)
            2: (-1, 0),   # esquerda
            3: (1, 0),    # direita
        }
        
        new_dir = action_map[action]
        # Impede inversão de 180° (cobra não pode virar do avesso)
        if (new_dir[0] + self.direction[0] != 0 or 
            new_dir[1] + self.direction[1] != 0):
            self.direction = new_dir

        # Calcula nova posição da cabeça
        head = self.snake[0]
        new_head = (head[0] + self.direction[0],
                    head[1] + self.direction[1])

        self.steps += 1
        self.steps_without_food += 1

        reward = 0.0
        done = False

        # ❌ Colisão com parede
        if (new_head[0] < 0 or new_head[0] >= self.GRID_SIZE or
            new_head[1] < 0 or new_head[1] >= self.GRID_SIZE):
            reward = -10.0
            done = True

        # ❌ Colisão com o próprio corpo
        elif new_head in self.snake:
            reward = -10.0
            done = True

        # ♻️ Limite de ciclos sem comer (evita loops infinitos)
        # Equivale a um 'timeout' — se a IA fica girando sem objetivo, para
        elif self.steps_without_food > self.GRID_SIZE * self.GRID_SIZE * 2:
            done = True
            reward = -5.0

        else:
            # Movimento válido — cobra avança
            self.snake.insert(0, new_head)  # nova cabeça

            # 🍎 Comeu a comida?
            if new_head == self.food:
                reward = 10.0          # +10: grande recompensa!
                self.score += 1
                self.steps_without_food = 0  # reseta contador
                self.food = self._place_food()  # nova comida
            else:
                self.snake.pop()  # remove a cauda (cobra só cresce ao comer)
                reward = 0.1      # +0.1: pequena recompensa por sobreviver

        return self._get_state(), reward, done

    def _get_state(self):
        """
        Constrói o vetor de estado — a 'foto' da situação atual.
        
        Imagine que é como o JSON que o servidor envia ao cliente:
        {
          headX: 0.5, headY: 0.3,
          dirX: 1, dirY: 0,
          foodDx: 0.2, foodDy: -0.1,
          dangerAhead: true, dangerRight: false, dangerLeft: false,
          bodyLength: 0.03
        }
        Mas como array: [0.5, 0.3, 1, 0, 0.2, -0.1, 1, 0, 0, 0.03]
        """
        head = self.snake[0]
        gs = self.GRID_SIZE

        # Posição normalizada (0 a 1, onde 0 = borda e 1 = borda oposta)
        head_x = head[0] / (gs - 1)
        head_y = head[1] / (gs - 1)

        # Direção atual (um 'vetor unitário' — qual eixo estamos nos movendo)
        dir_x = self.direction[0]
        dir_y = self.direction[1]

        # Posição relativa da comida (para onde olhar?)
        food_dx = (self.food[0] - head[0]) / (gs - 1)
        food_dy = (self.food[1] - head[1]) / (gs - 1)

        # Danger detection — sensor de perigo em 3 direções
        danger_straight = self._is_dangerous(self.direction)
        # Rotação 90° à direita (rotação de vetor: x,y → y,-x)
        dir_right = (self.direction[1], -self.direction[0])
        dir_left = (-self.direction[1], self.direction[0])
        danger_right = self._is_dangerous(dir_right)
        danger_left = self._is_dangerous(dir_left)

        # Comprimento da cobra relativo ao tabuleiro inteiro
        length_norm = len(self.snake) / (gs * gs)

        return np.array([
            head_x, head_y,
            dir_x, dir_y,
            food_dx, food_dy,
            float(danger_straight),
            float(danger_right),
            float(danger_left),
            length_norm,
        ], dtype=np.float32)

    def _is_dangerous(self, direction):
        """
        Verifica se mover numa dada direção causa colisão imediata.
        É como um 'sensor' — a IA não vê o tabuleiro, mas tem 'sensores'
        que detectam perigo nas 3 direções relativas.
        """
        head = self.snake[0]
        new_pos = (head[0] + direction[0], head[1] + direction[1])
        if (new_pos[0] < 0 or new_pos[0] >= self.GRID_SIZE or
            new_pos[1] < 0 or new_pos[1] >= self.GRID_SIZE):
            return 1.0  # parede
        if new_pos in self.snake:
            return 1.0  # corpo da própria cobra
        return 0.0  # livre

    def render(self, ax=None):
        """
        Desenha o tabuleiro num matplotlib Axes.
        Cores:
            fundo: preto    (#1a1a2e)
            corpo: verde escuro (#2d6a4f)
            cabeça: verde claro (#52b788)
            comida: rosa    (#ef476f)
        """
        if ax is None:
            fig, ax = plt.subplots(figsize=(4, 4))

        gs = self.GRID_SIZE
        grid = np.zeros((gs, gs), dtype=int)

        for x, y in self.snake[1:]:
            grid[y, x] = 1  # corpo
        hx, hy = self.snake[0]
        grid[hy, hx] = 2  # cabeça
        fx, fy = self.food
        grid[fy, fx] = 3  # comida

        cmap = matplotlib.colors.ListedColormap(
            ['#1a1a2e', '#2d6a4f', '#52b788', '#ef476f']
        )
        bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
        norm = matplotlib.colors.BoundaryNorm(bounds, cmap.N)

        ax.clear()
        ax.imshow(grid, cmap=cmap, norm=norm, interpolation='nearest')
        ax.set_title(f"Score: {self.score} | Steps: {self.steps}",
                     fontsize=12, fontweight='bold')
        ax.grid(True, color='gray', linewidth=0.3, alpha=0.3)
        ax.set_xticks([])
        ax.set_yticks([])

        return ax

# Parte 2: Rede Neural (DQN)

## O que é uma Rede Neural? (Explicação Detalhada)

Uma rede neural é, na prática, uma **cadeia de transformações matemáticas**. Cada transformação tem **parâmetros ajustáveis** (os "pesos"). É como uma composição de funções:

```
saída = f3(f2(f1(entrada)))
```

### Analogia com Middleware no Express.js

```javascript
// Cada middleware transforma a request:
app.use(parserJSON);      // camada 1: parseia dados brutos
app.use(auth);            // camada 2: extrai features do usuário
app.use(processarDados);  // camada 3: processamento final
app.post('/rota', handler); // output final
```

```python
# O mesmo conceito numa rede neural:
camada1 = ReLU(Linear(input))     # extrai features básicas
camada2 = ReLU(Linear(camada1))   # combina features
output  = Linear(camada2)         # resultado final
```

## O que é um "Linear Layer" (camada linear)?

É a operação mais simples de uma rede neural. Pense nisso como **multiplicar por uma matriz** — como uma transformação linear em gráficos:

```
# Exemplo simplificado:
# entrada: [x1, x2, x3]  (3 números)
# saída:   [y1, y2]      (2 números)

# A camada Linear tem uma matriz de pesos W (2×3):
#        [w11, w12, w13]
#    W = [w21, w22, w23]

# E um vetor de bias b (2):
#    b = [b1, b2]

# O cálculo é: y = x × W^T + b  (produto matricial)
# Isso é literalmente: cada output é uma combinação linear dos inputs
```

**Os pesos (W) e biases (b) começam como números aleatórios.** É o treino que vai ajustá-los.

**Visualmente** — nossa rede:

```
 10 inputs        128 neurônios      128 neurônios      4 outputs
 (estado)         (ocultos)          (ocultos)          (ações)
                                                          ↑
    ●              ● ● ● ... ●        ● ● ● ... ●    Q(cima)
    ●              ● ● ● ... ●   →    ● ● ● ... ● →  Q(baixo)
    ●              ● ● ● ... ●        ● ● ● ... ●    Q(esquerda)
    .              ● ● ● ... ●        ● ● ● ... ●    Q(direita)
    ●

 Cada linha entre colunas = pesos aprendidos durante o treino
```

## O que é ReLU (Rectified Linear Unit)?

É a função de ativação mais simples e popular:

```
ReLU(x) = max(0, x)

# Se o valor é positivo → passa direto
# Se o valor é negativo → vira zero
```

**Por que precisamos disso?** Sem ativação, todas as camadas lineares se colapsariam em UMA só matriz. A ativação introduz **não-linearidade**, que permite a rede aprender padrões complexos.

## O que é Q-Value?

**Q-Value** = "Qual o valor total (Q) de tomar esta ação neste estado?"

É como um **score de prioridade**:
- `Q(cima) = 5` → ir para cima é uma boa ideia (valor 5)
- `Q(baixo) = -3` → ir para baixo é ruim (valor -3)
- `Q(direita) = 8` → ir para direita é ótimo (valor 8)
- `Q(esquerda) = -10` → ir para esquerda é péssimo (provavelmente tem parede)

A rede **aprende esses valores** tentando e errando. No início são aleatórios, mas depois convergem.

## Nossa Arquitetura

```
Input (10) → ReLU → Hidden(128) → ReLU → Hidden(128) → Output(4)
```

- **10 entradas**: o estado do jogo
- **128 neurônios**: primeira camada oculta (extrai padrões)
- **128 neurônios**: segunda camada oculta (refina padrões)
- **4 saídas**: Q-value de cada ação (cima, baixo, esquerda, direita)

In [ ]:
# ============================================================
# DQN — Rede Neural (Deep Q-Network)
# ============================================================
class DQN(nn.Module):
    """
    Deep Q-Network: recebe estado → retorna Q-values das 4 ações.
    
    Em termos web dev, é como um service que recebe dados
    e retorna os 'scores' de cada ação possível.
    
    Arquitetura:
        Linear(10, 128) → ReLU → Linear(128, 128) → ReLU → Linear(128, 4)
    
    Exemplo de uso:
        entrada  = [0.5, 0.3, 1, 0, 0.2, -0.1, 1, 0, 0, 0.03]  # estado
        saída    = [5.2, -3.1, -10.0, 8.7]  # Q-values de (cima, baixo, esq, dir)
        decisão  = argmax(saída) = 3 (direita, que tem maior Q-value)
    """

    def __init__(self, input_dim=10, output_dim=4, hidden_dim=128):
        super().__init__()
        # Cada Linear é uma 'camada' — matriz de pesos + bias
        # Linear(10, 128) = matriz 128×10 + vetor bias de 128
        # Linear(128, 128) = matriz 128×128 (a 'camada do meio')
        # Linear(128, 4) = matriz 4×128 (uma linha por ação)
        self.fc1 = nn.Linear(input_dim, hidden_dim)    # 10 → 128
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)    # 128 → 128
        self.fc3 = nn.Linear(hidden_dim, output_dim)    # 128 → 4

    def forward(self, x):
        """
        Forward pass — o 'pipeline' de transformação dos dados.
        É como o pipe() do Express: cada camada processa e passa para a próxima.
        """
        x = F.relu(self.fc1(x))   # fc1 transforma, ReLU adiciona não-linearidade
        x = F.relu(self.fc2(x))   # segunda camada de processamento
        return self.fc3(x)        # output final: 4 Q-values (sem ativação)
        # Não usamos ativação no output porque queremos Q-values livres
        # (podem ser positivos, negativos, grandes, pequenos — sem limite)

# Parte 3: Agente DQN — O "Orquestrador" do Aprendizado

Se a rede neural é o "cérebro", o **agente** é o cérebro + sistema de memória + sistema de decisão. Ele coordena:

## 1. ε-Greedy (Exploração vs Exploração)

**Problema**: Num restaurante, você sempre vai no mesmo lugar que já sabe que é bom (exploração), ou tenta um novo que pode ser ainda melhor (exploração)?

O parâmetro **ε (epsilon)** controla isso:

```
ε = 1.0  →  100% exploração: ações 100% aleatórias (como um bebê aprendendo)
ε = 0.5  →  50% aleatório, 50% usa o que aprendeu
ε = 0.01 →  99% usa o que aprendeu, 1% tenta algo novo
```

No treino, **ε decai gradualmente**: começa em 1.0 e termina em 0.01.

```
Episódio 1:   ε = 1.000  ██████████ 100% aleatório (explora tudo)
Episódio 50:  ε = 0.780  ███████▊░░ 78% aleatório
Episódio 200: ε = 0.360  ███▌░░░░░░ 36% aleatório
Episódio 500: ε = 0.060  ░░░░░░░░░░ 6% aleatório (quase nada)
```

**Analogia web dev:** É como **feature flags** que mudam gradualmente. No início, você habilita tudo para testar. Depois, vai estabilizando.

## 2. Experience Replay (Memória de Experiências)

Cada ação gera uma **transição**: `(state, action, reward, next_state, done)`

Pense nisso como um **log** ou **audit trail**:
```python
# Exemplo de uma transição:
(
    [0.5, 0.3, 1, 0, 0.2, -0.1, 1, 0, 0, 0.03],   # onde estava
    3,                                              # o que fez (direita)
    0.1,                                            # o que ganhou
    [0.6, 0.3, 1, 0, 0.1, -0.1, 0, 0, 0, 0.03],    # onde ficou
    False                                           # morreu? não
)
```

Armazenamos até 50.000 transições e **sampleamos batches de 64** para treinar. Isso é como fazer **data science sobre o log** — aprendemos com o histórico ao invés de apenas o evento mais recente.

**Por que samplear ao invés de usar a experiência mais recente?**
- Se treinássemos só com a última experiência, a rede "esqueceria" o passado (como amnésia).
- Sampleando aleatoriamente, misturamos experiências antigas e novas → treino mais estável.

## 3. Double DQN — Duas Redes (Policy + Target)

Temos **DUAS redes neurais**:
- **Policy Network**: rede principal — é a que tomamos decisões e treinamos.
- **Target Network**: cópia congelada — serve como "referência estável" para calcular o target.

**Por que duas?** Se usássemos a mesma rede para decidir E avaliar, seria como "pegar prova com consulta aberta e ser o professor que corrige" — o modelo se auto-engana. A target network é a "versão congelada de você mesmo de 10 episódios atrás" — mais objetiva.

A cada 10 episódios, copiamos os pesos da Policy para a Target.

## 4. Backpropagation — Como a Rede "Aprende"

É assim que os pesos são ajustados:

```
1. Feed Forward:  passa os dados pela rede → obtém previsão
2. Calcula o Erro: compara previsão com o esperado (loss)
3. Backpropagation: calcula a 'culpa' de cada peso no erro
4. Atualiza Pesos: ajusta cada peso um pouquinho na direção que reduz o erro
5. Repete: faz isso milhares de vezes até o erro ficar pequeno
```

**Analogia:** Imagine ajustar manualmente os knobs de um equalizador.
- Cada knob = um peso da rede.
- Cada ajuste que você faz = backpropagation.
- O som que melhora = loss diminuindo.

O **optimizador Adam** é o "algoritmo" que decide o quanto mover cada peso. É uma versão inteligente de gradiente descendente — como um autotuner que encontra a afinação perfeita.

## 5. Huber Loss (Smooth L1 Loss)

É a função que mede o erro. É mais robusta que MSE (Mean Squared Error):

- Para erros pequenos: age como MSE (preciso)
- Para erros grandes: age como MAE (não explode com outliers)

**É como um try/catch:** se o erro é grande demais, não tenta corrigir tudo de uma vez.

## O Fluxo Completo

```
┌─────────────────────────────────────────────────────────┐
│                   AGENTE DQN                             │
│                                                          │
│  loop por episódio:                                      │
│    │                                                     │
│    ├─ ε-greedy → escolhe ação (aleatória ou pela rede)   │
│    │                                                     │
│    ├─ step(action) → obtém reward, next_state, done      │
│    │                                                     │
│    ├─ push() → salva no replay buffer (memory)           │
│    │                                                     │
│    ├─ train_step() → se buffer tem dados suficientes:    │
│    │     ├─ sampleia batch aleatório                     │
│    │     ├─ calcula target Q com Double DQN              │
│    │     ├─ calcula loss = SmoothL1(predição, target)    │
│    │     ├─ loss.backward() → backpropagation            │
│    │     └─ optimizer.step() → atualiza pesos            │
│    │                                                     │
│    ├─ decay_epsilon() → reduz exploração                 │
│    └─ update_target() → a cada N episódios,              │
│                         congela rede atual como target    │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# DQN AGENT — O cérebro completo do sistema
# ============================================================
class DQNAgent:
    """
    Agente DQN com todos os componentes de aprendizado.
    
    Pense nisso como a 'classe Controller' que gerencia:
    - A tomada de decisão (select_action)
    - A memória (replay buffer)
    - O treino (train_step)
    - A estabilidade (target network + epsilon decay)
    """

    def __init__(self, state_dim=10, action_dim=4, lr=1e-3,
                 gamma=0.99, buffer_size=50000, batch_size=64,
                 target_update=10, eps_start=1.0, eps_end=0.01,
                 eps_decay=0.995):
        """
        Inicializa o agente com todos os hiper-parâmetros.
        'Hiper-parâmetros' são configurações do algoritmo — como um .env:
        
        Parâmetros:
            state_dim     → tamanho do estado (10 números)
            action_dim    → número de ações possíveis (4: cima/baixo/esq/dir)
            lr            → learning rate: quão grande é cada ajuste nos pesos
                            Alto = aprende rápido, mas instável
                            Baixo = aprende devagar, mas estável
                            lr=1e-3 = 0.001 → ajuste sutil por vez
            gamma         → quanto valorizamos recompensas futuras vs imediatas
                            gamma=0.99 → valoriza 99% do futuro (visão de longo prazo)
            buffer_size   → tamanho máximo do replay buffer (deque circular)
            batch_size    → quantas experiências sampleamos por treino
            target_update → a cada quantos episódios atualizamos a target network
            eps_start/end → início e fim do epsilon (100% → 1% aleatório)
            eps_decay     → taxa de decaimento por episódio (0.995 = -0.5%/ep)
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma  # fator de desconto futuro
        self.batch_size = batch_size
        self.target_update = target_update
        self.eps = eps_start   # começa explorando tudo
        self.eps_end = eps_end  # termina quase sem exploração
        self.eps_decay = eps_decay

        # Duas redes: principal (treinável) e alvo (congelada)
        self.policy_net = DQN(state_dim, action_dim).to(self.device)
        self.target_net = DQN(state_dim, action_dim).to(self.device)
        # Inicialmente são idênticas — target vai ficar congelada
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()  # target em modo de inferência

        # Optimizer: o algoritmo que ajusta os pesos
        # Adam = adaptive moment estimation (o mais popular em Deep Learning)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        
        # Replay buffer: deque com tamanho máximo (fila circular)
        # Quando enche, descarta as experiências mais antigas automaticamente
        # Como um 'circular buffer' ou ring buffer em sistemas
        self.memory = deque(maxlen=buffer_size)

    def select_action(self, state):
        """
        Decide a próxima ação. Lógica epsilon-greedy:
        
        if random() < eps:
            # MODO EXPLORAÇÃO — ação aleatória (teste)
            return ação aleatória
        else:
            # MODO EXPLORAÇÃO — usa o que aprendeu
            passa state pela rede → pega ação com maior Q-value
        
        Analogia: ε = probabilidade de "tentar algo novo"
        - ε=1.0: tenta tudo aleatoriamente (como um bebê)
        - ε=0.01: usa expertise, quase nunca tenta algo novo
        """
        if random.random() < self.eps:
            return random.randint(0, self.action_dim - 1)  # exploração
        with torch.no_grad():  # sem gradiente = só inferência, sem backprop
            s = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(s)  # [Q(cima), Q(baixo), Q(esq), Q(dir)]
            return q_values.argmax(dim=1).item()  # índice do maior Q-value

    def push(self, state, action, reward, next_state, done):
        """
        Salva uma experiência na memória.
        Como gravar um log: (onde estava, o que fez, o que ganhou, onde ficou, acabou?)
        """
        self.memory.append((state, action, reward, next_state, done))

    def train_step(self):
        """
        UM passo de treino. É aqui que a mágica do aprendizado acontece.
        
        Equação do Double DQN (Bellman):
        
            Target Q = reward + gamma * Q_target(s', argmax Q_policy(s'))
        
        Em português claro:
            "O valor real de fazer a ação X é o que ganhei agora (reward)
             mais o que espero ganhar no futuro (Q do próximo estado),
             descontado pelo gamma."
        
        Depois comparamos o Q que a rede previu com esse Target Q,
        calculamos o erro (loss) e ajustamos os pesos da rede.
        """
        if len(self.memory) < self.batch_size:
            return 0.0  # buffer vazio demais, não treina

        # Sampleia um batch aleatório (como SELECT * ORDER BY RAND() LIMIT 64)
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        # Converte tudo para tensores PyTorch
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        # Q-values previstos pela rede principal
        # gather = "pegar o Q-value da ação que foi realmente tomada"
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Double DQN — calcula o Q target
        # A policy_net decide QUAL ação é melhor no próximo estado
        # A target_net avalia QUÃO boa é essa ação
        with torch.no_grad():  # não precisa de gradientes aqui
            next_actions = self.policy_net(next_states).argmax(dim=1)
            next_q = self.target_net(next_states).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)
            # Fórmula de Bellman: target = reward + gamma * Q_future
            # Se done=True, não tem futuro → target = reward
            target_q = rewards + (1 - dones) * self.gamma * next_q

        # Loss: quão errada é a previsão vs. o target?
        # Smooth L1 (Huber) = robusto a outliers
        loss = F.smooth_l1_loss(q_values, target_q)

        # Backpropagation: calcula gradientes e atualiza pesos
        self.optimizer.zero_grad()  # limpa gradientes anteriores
        loss.backward()             # calcula gradientes (backprop)
        torch.nn.utils.clip_grad_value_(
            self.policy_net.parameters(), 1.0
        )  # gradient clipping: evita atualizações explosivas
        self.optimizer.step()       # aplica as atualizações

        return loss.item()

    def update_target(self):
        """
        Copia os pesos da policy_net para target_net.
        É como 'congelar' a versão atual da rede para ter uma referência estável.
        """
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def decay_epsilon(self):
        """
        Decai epsilon: eps = eps * 0.995
        A cada episódio, ~0.5% menos exploração aleatória.
        Começa em 1.0 (100%), termina em 0.01 (1%) — o piso.
        """
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

# Parte 4: Treinamento com Feedback Visual

## O Loop de Treino

Agora juntamos tudo. O treino consiste em:

```
PARA cada episódio (1 até 500):
    1. Reinicia o jogo
    2. ENQUANTO jogo não acabou:
        a. Agente escolhe ação (epsilon-greedy)
        b. Ambiente executa e retorna (state, reward, done)
        c. Agente salva experiência no replay buffer
        d. Agente faz um step de treino (backpropagation)
    3. Após o episódio:
        a. Decai epsilon (menos aleatório)
        b. Atualiza target network a cada 10 episódios
        c. Log: score, loss, epsilon
        d. Salva modelo se for o melhor até agora
```

## O que você vai ver

- **Gráfico de score**: começa baixo (ações aleatórias) e sobe (aprendizado)
- **Gráfico de epsilon**: decaimento de 1.0 → 0.01
- **Demo visual**: a cada 50 episódios, vemos a IA jogar (melhora visualmente)
- **Console**: print a cada 50 episódios com métricas

## Hiper-parâmetros (o ".env" do treino)

| Parâmetro | Valor | Significado |
|---|---|---|
| `NUM_EPISODES` | 500 | Quantas partidas a IA joga para aprender |
| `RENDER_EVERY` | 50 | A cada quantos episódios mostra o jogo visualmente |
| `GRID_SIZE` | 10 | Tabuleiro 10×10 = 100 células |
| `lr` | 0.001 | Taxa de aprendizado — passos pequenos e seguros |
| `gamma` | 0.99 | Valoriza 99% do futuro (long-term planning) |
| `batch_size` | 64 | Sampleia 64 experiências por step de treino |
| `eps_start` | 1.0 | Começa 100% aleatório |
| `eps_decay` | 0.993 | De ~0.7% de redução por episódio |

In [ ]:
# ============================================================
# TREINAMENTO — Loop principal
# ============================================================

NUM_EPISODES = 500       # número de episódios de treino
RENDER_EVERY = 50        # renderiza o jogo a cada N episódios
GRID_SIZE = 10           # tabuleiro 10x10

# Cria ambiente e agente
env = SnakeGame(grid_size=GRID_SIZE)
agent = DQNAgent(
    state_dim=10,       # nosso estado tem 10 features
    action_dim=4,       # 4 ações possíveis
    lr=1e-3,            # learning rate
    gamma=0.99,         # desconto futuro
    buffer_size=50000,  # memórias de até 50k experiências
    batch_size=64,      # batch de treino
    target_update=10,   # atualiza target network a cada 10 episódios
    eps_start=1.0,      # começa 100% aleatório
    eps_end=0.01,       # termina 1% aleatório
    eps_decay=0.993,    # decaimento suave
)

# Arrays para logging (como métricas do DataDog)
scores = []
epsilons = []
losses = []
avg_scores = []

best_score = -1
start_time = time.time()

# Cria figura para plot em tempo real
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Training Progress", fontsize=14, fontweight='bold')

for episode in range(1, NUM_EPISODES + 1):
    state = env.reset()
    total_reward = 0
    episode_loss = 0
    steps_count = 0

    # Uma partida inteira
    while not env.game_over:
        action = agent.select_action(state)   # ε-greedy
        next_state, reward, done = env.step(action)  # ambiente responde
        agent.push(state, action, reward, next_state, done)  # salva memória
        loss = agent.train_step()              # backpropagation
        episode_loss += loss
        total_reward += reward
        steps_count += 1
        state = next_state
        if done:
            break

    # Fim do episódio — atualizações
    agent.decay_epsilon()  # reduz exploração
    if episode % agent.target_update == 0:
        agent.update_target()  # congela rede atual como target

    # Logging
    scores.append(env.score)
    epsilons.append(agent.eps)
    losses.append(episode_loss / max(steps_count, 1))
    window = min(50, len(scores))
    avg_scores.append(np.mean(scores[-window:]))  # média móvel de 50

    # Salva modelo se superou o recorde
    if env.score > best_score:
        best_score = env.score
        torch.save(agent.policy_net.state_dict(), "best_snake_model.pth")

    # Demo visual periódica — mostra a IA jogando sem aleatoriedade
    if episode % RENDER_EVERY == 0 or episode == 1:
        env2 = SnakeGame(grid_size=GRID_SIZE)
        demo_state = env2.reset()
        demo_done = False
        while not demo_done:
            old_eps = agent.eps
            agent.eps = 0.0  # força uso exclusivo da rede
            act = agent.select_action(demo_state)
            agent.eps = old_eps
            demo_state, _, demo_done = env2.step(act)
        render_ax = axes[0]
        env2.render(render_ax)
        render_ax.set_title(f"Demo Ep.{episode} | Score: {env2.score}")

    # Atualiza gráficos a cada 10 episódios
    if episode % 10 == 0 or episode == 1:
        axes[1].clear()
        axes[1].plot(scores, alpha=0.3, color='gray', label='raw')
        axes[1].plot(avg_scores, color='blue', linewidth=2, label='avg(50)')
        axes[1].set_title("Score por Episódio")
        axes[1].set_xlabel("Episódio")
        axes[1].set_ylabel("Score")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        axes[2].clear()
        axes[2].plot(epsilons, color='orange')
        axes[2].set_title("Epsilon (Exploração → Exploração)")
        axes[2].set_xlabel("Episódio")
        axes[2].set_ylabel("ε")
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        display(fig)
        clear_output(wait=True)

    # Print de progresso no console
    if episode % 50 == 0:
        elapsed = time.time() - start_time
        print(f"Ep {episode}/{NUM_EPISODES} | "
              f"Score: {env.score:3d} | Best: {best_score:3d} | "
              f"ε={agent.eps:.3f} | Time: {elapsed:.0f}s")

print("\n" + "=" * 50)
print(f"Treinamento concluído! Melhor score: {best_score}")
print(f"Modelo salvo em 'best_snake_model.pth'")

# Parte 5: Visualização do Agente Treinado

Agora carregamos o melhor modelo (salvo durante o treino) e assistimos a IA jogar.

Carregar o modelo é como carregar um arquivo `.json` de configuração — os pesos são números que foram ajustados durante o treino.

**Nesta fase, ε = 0** (zero aleatoriedade). A IA usa APENAS o que aprendeu.

Espere ver:
- Nos primeiros episódios do treino: cobra bate na parede, morre rápido
- Ao final do treino: cobra deve navegar até a comida com eficiência

In [ ]:
# ============================================================
# VISUALIZAÇÃO — Assista a IA jogar com o modelo treinado
# ============================================================

# Carrega os pesos do melhor modelo
# state_dict é como um JSON serializado — contem todos os pesos da rede
agent.policy_net.load_state_dict(torch.load(
    "best_snake_model.pth", weights_only=True, map_location=agent.device
))
# Modo .eval() = desativa comportamentos de treino, usa só inferência
agent.policy_net.eval()

NUM_GAMES = 5
results = []

for game in range(NUM_GAMES):
    env_test = SnakeGame(grid_size=GRID_SIZE)
    state = env_test.reset()
    done = False
    
    fig_test, ax_test = plt.subplots(figsize=(5, 5))
    
    while not done:
        # Inferência pura — sem aleatoriedade (ε = 0)
        with torch.no_grad():  # sem gradientes = mais rápido e seguro
            s = torch.FloatTensor(state).unsqueeze(0).to(agent.device)
            # A rede retorna [Q(cima), Q(baixo), Q(esq), Q(dir)]
            q_values = agent.policy_net(s)
            # Escolhe a ação com maior Q-value
            action = q_values.argmax(dim=1).item()
        
        state, _, done = env_test.step(action)
        
        env_test.render(ax_test)
        ax_test.set_title(
            f"Game {game+1}/{NUM_GAMES} | Score: {env_test.score}",
            fontsize=14, fontweight='bold'
        )
        plt.pause(0.05)  # pausa para animação (como um setTimeout visual)
    
    results.append(env_test.score)
    plt.close(fig_test)
    print(f"Game {game+1}: Score = {env_test.score}")

print(f"\nMédia de score: {np.mean(results):.1f}")
print(f"Melhor jogo:    {max(results)}")
print(f"Pior jogo:      {min(results)}")

# Parte 6: Gráficos de Análise Final

Análise pós-treino — como fazer um dashboard das métricas.

Os 4 gráficos que veremos:
1. **Score por Episódio** — a curva de aprendizado: deve ter tendência crescente
2. **Epsilon Decay** — de 100% para 1%: visualiza a transição exploração→exploitação
3. **Loss por Episódio** — deve cair (rede errando menos) e estabilizar
4. **Distribuição de Scores** — histograma: onde a maioria dos scores se concentra

In [ ]:
# ============================================================
# ANÁLISE FINAL — Dashboard de métricas
# ============================================================
plt.style.use('default')
fig, axes_final = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Análise Final do Treinamento", fontsize=16, fontweight='bold')

# 1. Score — a curva de aprendizado
ax = axes_final[0, 0]
ax.plot(scores, alpha=0.3, color='gray', label='Score por episódio')
ax.plot(avg_scores, color='steelblue', linewidth=2, label='Média móvel (50)')
ax.set_title("Evolução do Score — A IA está melhorando?")
ax.set_xlabel("Episódio")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True, alpha=0.3)
# Interprete: tendência de subida = aprendizado funcionando

# 2. Epsilon — transição de exploração para exploração
ax = axes_final[0, 1]
ax.plot(epsilons, color='darkorange')
ax.set_title("Decaimento de Epsilon\n(de 100% aleatório para 1% aleatório)")
ax.set_xlabel("Episódio")
ax.set_ylabel("ε")
ax.grid(True, alpha=0.3)

# 3. Loss — erro da rede por episódio
ax = axes_final[1, 0]
ax.plot(losses, color='crimson', alpha=0.7)
ax.set_title("Loss por Episódio\n(quanto menor, melhor a previsão)")
ax.set_xlabel("Episódio")
ax.set_ylabel("Smooth L1 Loss")
ax.grid(True, alpha=0.3)
# Interprete: queda = rede prevendo melhor; instabilidade = normal em RL

# 4. Distribuição de scores — onde a maioria fica
ax = axes_final[1, 1]
ax.hist(scores, bins=30, color='seagreen', edgecolor='white', alpha=0.8)
ax.set_title("Distribuição de Scores")
ax.set_xlabel("Score")
ax.set_ylabel("Frequência")
ax.axvline(np.mean(scores), color='red', linestyle='--',
           label=f'Média: {np.mean(scores):.1f}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resumo Final — Tudo Junto

## Arquitetura do Sistema

| Componente | O que faz | Analogia Web Dev |
|---|---|---|
| **SnakeGame** | Ambiente/Engine do jogo | O servidor que processa regras |
| **Estado (10D)** | Vetor com a situação | `req.body` — dados de entrada |
| **DQN (Rede)** | Prediz Q-value de cada ação | O service que toma decisões |
| **Replay Buffer** | Memória de experiências | Audit trail / log |
| **Target Network** | Cópia congelada da rede | Snapshot para comparação |
| **Epsilon** | Taxa de exploração | Rate de "tentar coisa nova" |
| **Optimizer Adam** | Ajusta pesos | O algoritmo de ajuste automático |
| **Loss (SmoothL1)** | Mede o erro | Métrica de qualidade |

## O Ciclo de Aprendizado (Passo a Passo)

```
FASE 1: INICIALIZAÇÃO (Episódio 1)
  ├─ Pesos da rede: aleatórios (como rand())
  ├─ Epsilon = 1.0 → 100% ações aleatórias
  └─ Resultado: cobra bate na parede imediatamente

FASE 2: EXPLORAÇÃO INTENSA (Episódios 1-50)
  ├─ ε cai de 1.0 → ~0.7
  ├─ Cobra ainda morre muito, mas ocasionalmente come
  ├─ Cada vez que come (+10), a rede ajusta pesos:
  │    "ah, naquela situação, ir para a comida foi bom"
  └─ Cada vez que morre (-10):
        "ah, naquela situação, ir para parede foi ruim"

FASE 3: APRENDIZADO TANGÍVEL (Episódios 50-200)
  ├─ ε cai de ~0.7 → ~0.3
  ├─ Cobra começa a navegar até a comida
  ├─ Já evita paredes na maioria das vezes
  └─ Score médio começa a subir consistentemente

FASE 4: REFINAMENTO (Episódios 200-500)
  ├─ ε cai de ~0.3 → ~0.01
  ├─ Cobra joga "sério" — quase sem aleatoriedade
  ├─ Deve conseguir scores de 5-15+ por partida
  └─ Aprende estratégias mais complexas:
       - Não ficar presa em cantos
       - Fugir do próprio corpo quando grande
```

## Glossário Completo

| Termo | Explicação Simples |
|---|---|
| **Tensor** | Array numérico otimizado para GPU (como um Float32Array) |
| **Neural Network** | Função com parâmetros ajustáveis via exemplo |
| **Forward Pass** | Passar dados pela rede (input → output) |
| **Backward Pass** | Calcular culpa de cada peso no erro |
| **Gradient** | Direção e magnitude de ajuste de cada peso |
| **Learning Rate** | O quanto ajustamos por vez — passos grandes ou pequenos |
| **Batch** | Grupo de exemplos processados juntos |
| **Epoch** | Uma passada completa por todos os dados de treino |
| **Overfitting** | Modelo decora os exemplos mas não generaliza |
| **Inference** | Usar o modelo treinado para prever (sem treino) |
| **state_dict** | Dicionário com todos os pesos — o "modelo salvo" |

## Melhorias Possíveis

- **Grid maior** (15×15, 20×20) — mais desafios
- **Input como imagem** — usar CNN ao invés de vetor (como olhar a tela)
- **Reward shaping** — recompensas mais sofisticadas
- **PPO/SAC** — algoritmos de RL mais avançados
- **Visual interativo** — widget para jogar contra a IA